# Assignment 2: Milestone I Natural Language Processing
## Task 2&3
#### Student Name: XXXX XXXX
#### Student ID: 000000


Environment: Python 3 and Jupyter notebook

Libraries used: please include all the libraries you used in your assignment, e.g.,:
* pandas
* re
* numpy

## Introduction
You should give a brief information of this assessment task here.

<span style="color: red"> Note that this is a sample notebook only. You will need to fill in the proper markdown and code blocks. You might also want to make necessary changes to the structure to meet your own needs. Note also that any generic comments written in this notebook are to be removed and replace with your own words.</span>

## Importing libraries 

In [ ]:
# Code to import libraries as you need in this assessment, e.g.,
import os
import numpy as np
import pandas as pd

from scipy.sparse import csr_matrix
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from IPython.display import display # Required to print neat Pandas tables

## Task 2. Generating Feature Representations for Clothing Items Reviews

For Task 2a, I generated bag-of-words count vectors using only `review_text` from `processed.csv`.
I loaded the vocabulary from `vocab.txt` created in Task 1, where each word has a fixed integer index.
For each review, I counted unigram frequencies and wrote a sparse vector line in the required format:

`#review_index,word_index:word_freq,word_index:word_freq,...`

The output was saved as `outputs/count_vectors.txt`, with one line per review.

### 1) Import + paths

In [ ]:
from pathlib import Path
import pandas as pd
from collections import Counter

OUTPUT_DIR = Path("../outputs")  # based on your Task 1 save location
processed_path = OUTPUT_DIR / "processed.csv"
vocab_path = OUTPUT_DIR / "vocab.txt"
count_vec_path = OUTPUT_DIR / "count_vectors.txt"

print("processed.csv:", processed_path.resolve(), processed_path.exists())
print("vocab.txt    :", vocab_path.resolve(), vocab_path.exists())

### 2) Load vocabulary (word -> index)

In [ ]:
# Build mapping: word -> integer index
word2idx = {}

with open(vocab_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        # Required format from Task 1: word:index
        word, idx = line.rsplit(":", 1)
        word2idx[word] = int(idx)

print("Vocabulary size:", len(word2idx))
print("First 10 vocab entries:", list(word2idx.items())[:10])

### 3) Load processed reviews

In [ ]:
df = pd.read_csv(processed_path)

# Task requirement: use review description/text only (ignore title)
reviews = df["review_text"].fillna("").astype(str).tolist()

print("Number of reviews:", len(reviews))
print("Example review:", reviews[0][:180], "...")

### 4) Convert each review into sparse count vector line

In [ ]:
def to_sparse_count_line(review_idx: int, review_text: str, w2i: dict[str, int]) -> str:
    """
    Convert one review into required sparse format:
    #<review_idx>,<word_idx>:<freq>,<word_idx>:<freq>,...
    """
    tokens = review_text.split() if review_text else []

    # Count only words present in Task 1 vocabulary
    freq = Counter(tok for tok in tokens if tok in w2i)

    # Sort by integer word index for stable output
    pairs = sorted((w2i[word], count) for word, count in freq.items())

    if not pairs:
        return f"#{review_idx},"

    sparse_part = ",".join(f"{idx}:{count}" for idx, count in pairs)
    return f"#{review_idx},{sparse_part}"


count_vector_lines = [
    to_sparse_count_line(i, text, word2idx)
    for i, text in enumerate(reviews)  # uses row index 0..N-1
]

print("First 3 lines preview:")
for line in count_vector_lines[:3]:
    print(line[:220])

### 5) Save count_vectors.txt

In [ ]:
# Ensure output folder exists
DATA_DIR.mkdir(parents=True, exist_ok=True)

with open(count_vec_path, "w", encoding="utf-8") as f:
    f.write("\n".join(count_vector_lines))

print("Saved:", count_vec_path.resolve())
print("Exists:", count_vec_path.exists())
print("Total lines written:", len(count_vector_lines))

### Saving outputs
Save the count vector representation as per spectification.
- count_vectors.txt

In [ ]:
# code to save output data...

## Task 3. Clothing Review Classification

...... Sections and code blocks on buidling classification models based on different document feature represetations. 
Detailed comparsions and evaluations on different models to answer each question as per specification. 

<span style="color: red"> You might have complex notebook structure in this section, please feel free to create your own notebook structure. </span>

### Q1: Language Model Comparisons

**Objective:** In this section, we compare the performance of different feature representations built in Task 2 (Bag-of-Words/Count, Unweighted Embeddings, and TF-IDF Weighted Embeddings). We will evaluate these representations using a Logistic Regression model and a Linear SVM classifier, utilizing 5-fold cross-validation to ensure reliable metrics.

#### 1. Settings & Configurations
Here we define our file paths and global settings. A provisional baseline is included to allow this notebook to run independently while Task 2 file outputs (`count_vectors.txt`, etc.) are being finalized by the team.

In [ ]:
DATA_PATH = "../data/cosmetics_beauty_products_reviews.csv"

# Switch this to True when Task 2 files are ready
USE_TASK2_FILES = False

COUNT_PATH = "count_vectors.txt"
WEIGHTED_PATH = "weighted_vectors.txt"
UNWEIGHTED_PATH = "unweighted_vectors.txt"

# Temporary baseline option for development
RUN_PROVISIONAL_COUNT_BASELINE = True

RANDOM_STATE = 42

#### 2. Load Dataset and Preprocess Labels
We load the raw dataset to extract the target variable (`is_a_buyer`). We handle potential formatting inconsistencies in the boolean column and map them strictly to integers (1 for True, 0 for False). We retain the original row index so we can safely align these labels with the custom document indices generated in the Task 2 text files.

In [ ]:
# Load dataset and labels
df = pd.read_csv(DATA_PATH).reset_index(drop=True)

# Keep raw row order unchanged so later Task 2 indices can match this dataframe
df["review_text"] = df["review_text"].fillna("")
df["review_title"] = df["review_title"].fillna("")

# label conversion to handle strings or booleans
if df["is_a_buyer"].dtype == bool:
    df["label"] = df["is_a_buyer"].astype(int)
else:
    df["label"] = (
        df["is_a_buyer"]
        .astype(str)
        .str.strip()
        .str.lower()
        .map({"true": 1, "false": 0})
    )

# Remove rows with unknown labels, if any
df = df.dropna(subset=["label"]).reset_index(drop=True)
df["label"] = df["label"].astype(int)

print("Dataset shape:", df.shape)
print("\nLabel distribution:")
print(df["label"].value_counts())
print("\nLabel proportion:")
print(df["label"].value_counts(normalize=True))

#### 3. Helper Functions for Data Parsing and Evaluation
The following functions parse the specific `.txt` formats outputted by Task 2.
* `load_count_vectors`: Reads the sparse representation into a highly efficient `scipy.sparse.csr_matrix` to save memory.
* `load_dense_vectors`: Reads the continuous sequence of values for embedding models.
* `evaluate_representation`: Handles the 5-fold Stratified Cross-Validation and extracts standard classification metrics.

In [ ]:
#  Helper functions

def load_count_vectors(path):
    """
    Reads sparse count vectors from file format:
    #doc_index,word_index:freq,word_index:freq,...
    Returns:
        X : csr_matrix
        doc_indices : np.array of original document indices
    """
    doc_indices = []
    rows = []
    cols = []
    data = []
    max_col = -1

    with open(path, "r", encoding="utf-8") as f:
        # Loop through every line (which represents one review) in the text file
        for row_id, line in enumerate(f):
            line = line.strip()
            if not line:
                continue

            # Split the line by commas. The first part is the document ID.
            parts = line.split(",")
            doc_idx = int(parts[0].replace("#", "").strip())
            doc_indices.append(doc_idx)

            # Loop through the rest of the parts (word_index:frequency)
            for item in parts[1:]:
                item = item.strip()
                if not item or ":" not in item:
                    continue
                # Split by colon to get the word's ID and how many times it appeared
                col_idx, value = item.split(":")
                col_idx = int(col_idx)
                # The frequency of the word
                value = float(value)

                # Store the coordinates (row, column) and the actual value
                rows.append(row_id)
                cols.append(col_idx)
                data.append(value)

                # Keep track of the highest word index to size our matrix properly
                if col_idx > max_col:
                    max_col = col_idx

    if len(doc_indices) == 0:
        return None, None

    # Construct a Compressed Sparse Row (CSR) matrix.
    # This only stores non-zero values, saving massive amounts of RAM.
    X = csr_matrix((data, (rows, cols)), shape=(len(doc_indices), max_col + 1))
    return X, np.array(doc_indices)


def load_dense_vectors(path):
    """
    Reads dense vectors from file format:
    #doc_index,val1,val2,val3,...
    Returns:
        X : np.ndarray
        doc_indices : np.array of original document indices
    """
    doc_indices = []
    vectors = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            # Extract the document ID and the sequence of float values
            parts = line.split(",")
            doc_idx = int(parts[0].replace("#", "").strip())
            values = [float(x) for x in parts[1:] if x.strip() != ""]

            doc_indices.append(doc_idx)
            vectors.append(values)

    if len(doc_indices) == 0:
        return None, None

    # Convert the list of lists into a standard numpy array
    X = np.array(vectors, dtype=float)
    return X, np.array(doc_indices)


def evaluate_representation(X, y, representation_name, classifier_name, classifier):
    """
    Runs 5-fold stratified cross-validation and returns summary metrics.
    """
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    # We tell sklearn to track 4 different metrics, not just accuracy
    scoring = {
        "accuracy": "accuracy",
        "precision": "precision",
        "recall": "recall",
        "f1": "f1"
    }

    scores = cross_validate(
        classifier,
        X,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        error_score="raise"
    )

    return {
        "Representation": representation_name,
        "Classifier": classifier_name,
        "Accuracy Mean": scores["test_accuracy"].mean(),
        "Accuracy Std": scores["test_accuracy"].std(),
        "Precision Mean": scores["test_precision"].mean(),
        "Recall Mean": scores["test_recall"].mean(),
        "F1 Mean": scores["test_f1"].mean()
    }

#### 4. Classifier Initialization & Representation Loading
We instantiate our chosen models (Logistic Regression and Linear SVM). Then, we load the respective feature representations.

In [ ]:
# Classifiers for Q1

classifiers = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        solver="liblinear",
        random_state=RANDOM_STATE
    ),
    "Linear SVM": LinearSVC(
        random_state=RANDOM_STATE
    ),
    "Naive Bayes": MultinomialNB()
}

# Prepare representations

representations = {}

# A. Optional provisional baseline (runs without Task 2 dependencies)
if RUN_PROVISIONAL_COUNT_BASELINE:
    count_vectorizer = CountVectorizer(max_features=5000)
    X_count_baseline = count_vectorizer.fit_transform(df["review_text"])
    representations["Count (provisional raw-text baseline)"] = {
        "X": X_count_baseline,
        "doc_indices": df.index.to_numpy()
    }

# B. Real Task 2 representations
if USE_TASK2_FILES:
    # Count vectors
    if os.path.exists(COUNT_PATH):
        X_count, idx_count = load_count_vectors(COUNT_PATH)
        representations["Count"] = {
            "X": X_count,
            "doc_indices": idx_count
        }
    else:
        print(f"[Warning] File not found: {COUNT_PATH}")

    # Weighted vectors
    if os.path.exists(WEIGHTED_PATH):
        X_weighted, idx_weighted = load_dense_vectors(WEIGHTED_PATH)
        representations["Weighted"] = {
            "X": X_weighted,
            "doc_indices": idx_weighted
        }
    else:
        print(f"[Warning] File not found: {WEIGHTED_PATH}")

    # Unweighted vectors
    if os.path.exists(UNWEIGHTED_PATH):
        X_unweighted, idx_unweighted = load_dense_vectors(UNWEIGHTED_PATH)
        representations["Unweighted"] = {
            "X": X_unweighted,
            "doc_indices": idx_unweighted
        }
    else:
        print(f"[Warning] File not found: {UNWEIGHTED_PATH}")

#### 5. Execute Q1 Experiments
Iterate through available language models, align the loaded document indices with the true labels, and evaluate performance via cross-validation.

In [ ]:
# Run Q1 experiments
results = []

for rep_name, rep_info in representations.items():
    X_rep = rep_info["X"]
    doc_idx = rep_info["doc_indices"]

    if X_rep is None or doc_idx is None:
        print(f"[Skip] {rep_name} could not be loaded.")
        continue

    # Align labels with the vector file document indices
    y_rep = df.loc[doc_idx, "label"].to_numpy()

    print(f"\nRunning experiments for: {rep_name}")
    print("X shape:", X_rep.shape)
    print("y shape:", y_rep.shape)

    for clf_name, clf in classifiers.items():
        print(f"  -> {clf_name}")
        row = evaluate_representation(
            X=X_rep,
            y=y_rep,
            representation_name=rep_name,
            classifier_name=clf_name,
            classifier=clf
        )
        results.append(row)

#### 6. Q1 Results Summary

In [ ]:
# Show result table
results_df = pd.DataFrame(results)

if not results_df.empty:
    results_df = results_df.sort_values(
        by=["F1 Mean", "Accuracy Mean"],
        ascending=False
    ).reset_index(drop=True)

    print("\nQ1 Results:")
    display(results_df)
else:
    print("No results to display. Check your file paths or baseline settings.")

### Q2: Does more information provide higher accuracy?

**Objective:** Q2 investigates whether adding supplementary product information improves the model's ability to predict purchasing behavior (`is_a_buyer`) compared to using the review text alone.

**Methodology:**
To rigorously test this while avoiding workflow bottlenecks, we use a hybrid evaluation approach:
1. **Text Only:** We reuse the exact Task 2 outputs (Bag-of-Words, TF-IDF Weighted, and Unweighted representations) generated by our teammates.
2. **Text + Title & Text + Title + Extra Info:** We dynamically build `sklearn.compose.ColumnTransformer` pipelines to generate these expanded feature sets on the fly. We mirror the Task 2 representations by utilizing `CountVectorizer`, `TfidfVectorizer`, and a binary `CountVectorizer` (as a mathematically equivalent proxy for unweighted embeddings).

All scenarios are evaluated using the same 5-fold Stratified Cross-Validation loop to ensure any gain in accuracy is strictly due to the addition of information.

#### 1. Data Loading and Cleaning (The Clean-up Crew)
We load a fresh instance of the dataset specifically for Q2. We ensure all text columns are free of `NaN` values and rigorously convert our boolean target label into a stable integer format (1s and 0s).

In [ ]:
# Load dataframe and labels
df_q2 = pd.read_csv(DATA_PATH).copy()

# Fill missing text columns with empty strings so the vectorizers don't crash
text_columns = ["review_text", "review_title", "product_title", "brand_name"]
for col in text_columns:
    if col not in df_q2.columns:
        df_q2[col] = "" # Create column if it doesn't exist to prevent KeyErrors
    df_q2[col] = df_q2[col].fillna("")

# Convert numeric columns to actual numbers, forcing errors into NaN
candidate_numeric = ["price", "avg_product_rating", "product_rating_count"]
for col in candidate_numeric:
    if col not in df_q2.columns:
        df_q2[col] = np.nan
    df_q2[col] = pd.to_numeric(df_q2[col], errors="coerce")

# Safely convert the target label into integers (1 for True, 0 for False)
if df_q2["is_a_buyer"].dtype == bool:
    df_q2["label"] = df_q2["is_a_buyer"].astype(int)
else:
    df_q2["label"] = (
        df_q2["is_a_buyer"]
        .astype(str)
        .str.strip()
        .str.lower()
        .map({"true": 1, "false": 0})
    )

# Drop any rows where the label is completely missing, then reset the index
df_q2 = df_q2.dropna(subset=["label"]).reset_index(drop=True)
df_q2["label"] = df_q2["label"].astype(int)

# Separate features (X) from target labels (y)
y_q2 = df_q2["label"].to_numpy()

print("Q2 dataframe shape:", df_q2.shape)
print("\nLabel distribution:\n", df_q2["label"].value_counts())

#### 2. Task 2 File Translators
To integrate our team's previously generated text files, we use custom parsers. Crucially, the `align_to_dataframe` function acts as a safety mechanism: it uses the extracted document indices to ensure the pre-computed matrices perfectly align with the current row order of our Q2 dataframe, preventing catastrophic data/label mismatch.

In [ ]:
#  Helper functions to read Task 2 files
def load_count_vectors(path):
    """ Reads sparse count vectors from Task 2 format into a CSR Sparse Matrix. """
    doc_indices = []
    rows, cols, data = [], [], []
    max_col = -1

    with open(path, "r", encoding="utf-8") as f:
        for row_id, line in enumerate(f):
            line = line.strip()
            if not line: continue

            # Extract the Document ID
            parts = line.split(",")
            doc_idx = int(parts[0].replace("#", "").strip())
            doc_indices.append(doc_idx)

            # Extract the coordinates and word frequencies
            for item in parts[1:]:
                item = item.strip()
                if not item or ":" not in item: continue
                c, v = item.split(":")
                c, v = int(c), float(v)

                rows.append(row_id)
                cols.append(c)
                data.append(v)
                if c > max_col: max_col = c

    if len(doc_indices) == 0: return None, None
    X = csr_matrix((data, (rows, cols)), shape=(len(doc_indices), max_col + 1))
    return X, np.array(doc_indices)


def load_dense_vectors(path):
    """ Reads sequence vectors from Task 2 format into a dense NumPy array. """
    doc_indices = []
    vectors = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line: continue

            parts = line.split(",")
            doc_idx = int(parts[0].replace("#", "").strip())
            # Convert the rest of the line into decimal numbers
            values = [float(x) for x in parts[1:] if x.strip() != ""]

            doc_indices.append(doc_idx)
            vectors.append(values)

    if len(doc_indices) == 0: return None, None
    X = np.array(vectors, dtype=float)
    return X, np.array(doc_indices)


def align_to_dataframe(X, doc_indices, n_rows):
    """ Safely maps the precomputed math matrix back to the exact row order of our CSV. """
    if X is None or doc_indices is None: return None

    # Ensure the matrix isn't missing any rows compared to our dataframe
    if len(doc_indices) != n_rows:
        raise ValueError(f"Vector file has {len(doc_indices)} rows, but dataframe has {n_rows} rows.")

    # Reorder the matrix to match the 0 to N-1 order of the dataframe
    order = np.argsort(doc_indices)
    if hasattr(X, "tocsr"):
        X = X[order] # For Sparse Matrices
    else:
        X = X[order, :] # For NumPy Arrays

    return X

#### 3. Cross-Validation Engines & Pipeline Factory
Here we define our judges. `evaluate_matrix` tests the precomputed Task 2 files. `evaluate_pipeline` tests our dynamically built Q2 scenarios. The `make_preprocessor` function acts as a factory, building the required `ColumnTransformer` (assembly line) on demand based on whether we are testing Bag-of-Words, Weighted, or Unweighted concepts.

In [ ]:
# Evaluation Helpers & Pipeline Factory
# Lock the folding strategy to be identical across all tests
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# --- Evaluator 1: For precomputed Task 2 files ---
def evaluate_matrix(X, y, representation_name, info_setting, classifier_name, classifier):
    scoring = {"accuracy": "accuracy", "precision": "precision", "recall": "recall", "f1": "f1"}
    scores = cross_validate(classifier, X, y, cv=cv, scoring=scoring, n_jobs=-1, error_score="raise")
    return {
        "Information Setting": info_setting,
        "Representation": representation_name,
        "Classifier": classifier_name,
        "Accuracy Mean": scores["test_accuracy"].mean(),
        "F1 Mean": scores["test_f1"].mean()
    }

# --- Evaluator 2: For dynamic Q2 Pipelines ---
def evaluate_pipeline(preprocessor, X_df, y, representation_name, info_setting, classifier_name, classifier):
    # Connect the data-assembly line to the classifier brain
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", classifier)
    ])
    scoring = {"accuracy": "accuracy", "precision": "precision", "recall": "recall", "f1": "f1"}
    scores = cross_validate(pipe, X_df, y, cv=cv, scoring=scoring, n_jobs=-1, error_score="raise")
    return {
        "Information Setting": info_setting,
        "Representation": representation_name,
        "Classifier": classifier_name,
        "Accuracy Mean": scores["test_accuracy"].mean(),
        "F1 Mean": scores["test_f1"].mean()
    }

# --- The Pipeline Factory ---
numeric_features = ["price", "avg_product_rating", "product_rating_count"]
# Fills missing prices with median, and scales everything to be mathematically equal
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler(with_mean=False))
])

def make_preprocessor(representation_name, include_extra):
    """ Dynamically builds a ColumnTransformer based on the requested scenario. """

    # Select the mathematical translator based on the representation name
    if representation_name == "Bag-of-Words":
        text_vec_main = CountVectorizer(max_features=5000)
        text_vec_title = CountVectorizer(max_features=2000)
        text_vec_product = CountVectorizer(max_features=2000)

    elif representation_name == "Weighted":
        text_vec_main = TfidfVectorizer(max_features=5000)
        text_vec_title = TfidfVectorizer(max_features=2000)
        text_vec_product = TfidfVectorizer(max_features=2000)

    elif representation_name == "Unweighted":
        # binary=True ignores frequency. It only checks "Did the word appear? 1 or 0"
        text_vec_main = CountVectorizer(max_features=5000, binary=True)
        text_vec_title = CountVectorizer(max_features=2000, binary=True)
        text_vec_product = CountVectorizer(max_features=2000, binary=True)

    # Base transformer always includes Text and Title
    transformers = [
        ("review_text_vec", text_vec_main, "review_text"),
        ("review_title_vec", text_vec_title, "review_title")
    ]

    # If scenario 3, append the extra metadata to the transformer
    if include_extra:
        transformers.extend([
            ("product_title_vec", text_vec_product, "product_title"),
            ("brand_ohe", OneHotEncoder(handle_unknown="ignore"), ["brand_name"]),
            ("numeric", numeric_transformer, numeric_features)
        ])

    return ColumnTransformer(transformers=transformers, remainder="drop")

# Define the models we are testing
classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=2000, solver="liblinear", random_state=RANDOM_STATE),
    "Linear SVM": LinearSVC(random_state=RANDOM_STATE)
}

#### 4. Execute Experiment Scenario 1: Text Only
We ingest the pre-computed files from Task 2 to serve as our foundation.

In [ ]:
# Q2 Part A: Text only (Task 2 files)

results = []
n_rows = len(df_q2)
text_only_representations = {}

# Safely load all three Task 2 files if they exist
if os.path.exists(COUNT_PATH):
    X_count_raw, idx_count = load_count_vectors(COUNT_PATH)
    text_only_representations["Bag-of-Words"] = align_to_dataframe(X_count_raw, idx_count, n_rows)
else: print("[Skip] count_vectors.txt not found.")

if os.path.exists(WEIGHTED_PATH):
    X_weighted_raw, idx_weighted = load_dense_vectors(WEIGHTED_PATH)
    text_only_representations["Weighted"] = align_to_dataframe(X_weighted_raw, idx_weighted, n_rows)
else: print("[Skip] weighted_vectors.txt not found.")

if os.path.exists(UNWEIGHTED_PATH):
    X_unweighted_raw, idx_unweighted = load_dense_vectors(UNWEIGHTED_PATH)
    text_only_representations["Unweighted"] = align_to_dataframe(X_unweighted_raw, idx_unweighted, n_rows)
else: print("[Skip] unweighted_vectors.txt not found.")

# Run the 5-fold CV for the loaded matrices
for rep_name, X_rep in text_only_representations.items():
    print(f"Running Text Only: {rep_name}...")
    for clf_name, clf in classifiers.items():
        row = evaluate_matrix(
            X=X_rep, y=y_q2,
            representation_name=rep_name, info_setting="Text only",
            classifier_name=clf_name, classifier=clf
        )
        results.append(row)

#### 5. Execute Experiment Scenarios 2 & 3: Information Expansion
We now utilize our pipeline factory to build and test the "Text + Title" and "Text + Title + Extra Info" settings across all three mathematical representations.

In [ ]:
# Q2 Part B & C: Adding Title & Extra Data

# Extract the raw features to pass into our pipelines
X_q2_df = df_q2[["review_text", "review_title", "product_title", "brand_name"] + numeric_features].copy()
q2_representation_names = ["Bag-of-Words", "Weighted", "Unweighted"]

# Scenario 2: Text + Title
print("\n--- Running Scenario 2: Text + Title ---")
for rep_name in q2_representation_names:
    print(f"Building pipeline for {rep_name}...")
    preprocessor = make_preprocessor(rep_name, include_extra=False)

    for clf_name, clf in classifiers.items():
        row = evaluate_pipeline(
            preprocessor=preprocessor, X_df=X_q2_df, y=y_q2,
            representation_name=rep_name, info_setting="Text + Title",
            classifier_name=clf_name, classifier=clf
        )
        results.append(row)

# Scenario 3: Text + Title + Extra Info
print("\n--- Running Scenario 3: Text + Title + Extra Info ---")
for rep_name in q2_representation_names:
    print(f"Building pipeline for {rep_name}...")
    preprocessor = make_preprocessor(rep_name, include_extra=True)

    for clf_name, clf in classifiers.items():
        row = evaluate_pipeline(
            preprocessor=preprocessor, X_df=X_q2_df, y=y_q2,
            representation_name=rep_name, info_setting="Text + Title + Extra",
            classifier_name=clf_name, classifier=clf
        )
        results.append(row)

#### 6. Q2 Final Results Summary

In [ ]:
# Final Q2 Table
q2_results_df = pd.DataFrame(results)

if not q2_results_df.empty:
    # Sort first by the amount of information, then the math representation, then highest F1 score
    q2_results_df = q2_results_df.sort_values(
        by=["Information Setting", "Representation", "F1 Mean"],
        ascending=[True, True, False]
    ).reset_index(drop=True)

print("\n===== Q2 RESULTS =====")
display(q2_results_df)

### Q2 Analysis and Findings

**Conclusion: Does more information provide higher accuracy?**
[Yes/No] - Based on our 5-fold cross-validation results, incorporating supplementary data [significantly improved / did not meaningfully improve] the model's ability to classify purchasing behavior compared to utilizing the baseline review descriptions alone.

**Key Observations:**
1. **The Impact of Titles (Scenario 2):** Combining the review title with the body text resulted in a [slight/significant] boost in the F1 score across classifiers. This suggests that titles often contain highly polarized, dense sentiment that aids classification.
2. **The Impact of Structured Metadata (Scenario 3):** Incorporating the product title, brand names (via One-Hot Encoding), and scaled numerical data (price and average rating) [further boosted / had little effect on] predictive performance.
3. **Representation Consistency:** Regardless of the information setting, the **[Insert winning representation, e.g., TF-IDF Weighted]** representation paired with the **[Insert winning classifier, e.g., Linear SVM]** consistently yielded the highest F1-scores, proving that [insert reason, e.g., TF-IDF scaling remains highly effective even when fusing text with numeric data].

## Summary
Give a short summary and anything you would like to talk about the assessment tasks here.

## Couple of notes for all code blocks in this notebook
- please provide proper comment on your code
- Please re-start and run all cells to make sure codes are runable and include your output in the submission.   
<span style="color: red"> This markdown block can be removed once the task is completed. </span>